# Quadruped Locomotion (Go2): Training (Fine-tuning)

이 노트북은 `go2_locomotion_basic.ipynb` 와 동일한 Colab 환경에서,
**env3 보상으로 학습해 둔 모델(`models/pretrained_env3`)을 불러와 4,800 step 만 추가 학습**한 뒤,
학습한 모델로 추론하여 보행 영상을 생성·재생합니다.

Colab 에서 전체 학습은 시간이 오래 걸리므로, 이어학습(fine-tuning)
방식으로 **학습 → 추론 → 영상** 파이프라인 전체를 빠르게 체험하는 것이 목적입니다.

학습/추론에 사용하는 보상 설정은 `src/envs3.yaml` 입니다.

> 런타임 → 런타임 유형 변경 → **T4 GPU** 로 설정 후 실행하세요.


---

## 0. 환경 설정

GitHub 레포지토리를 clone 하고, MuJoCo / Stable-Baselines3 등 의존성을 설치합니다.
Colab 이면 `/content` 를, 아니면 현재 작업 디렉터리를 기준 경로로 사용합니다.

> ⚠️ 설치 후 numpy 적용을 위해 **최초 1회 런타임이 자동 재시작**됩니다(잠깐 끊김 = 정상). 다시 연결되면 [런타임 > 모두 실행]만 누르면 되고, 그 다음부터는 재시작이 없습니다.


In [ ]:
# 1) Clone repository
import os, sys

# Detect Colab by availability of /content or google.colab.
try:
    import google.colab  # noqa: F401
    in_colab = True
except Exception:
    in_colab = os.path.isdir("/content")

try:
    base_dir
except NameError:
    base_dir = "/content" if in_colab else os.getcwd()
os.chdir(base_dir)

repo_dir = os.path.join(base_dir, "RL_tutorial")

print(f"Base directory: {base_dir}")
print(f"Repo directory: {repo_dir}")

if not os.path.isdir(repo_dir):
  !git clone https://github.com/agbread/RL_tutorial.git
else:
  print("Cloned Directory already exists")

os.chdir(repo_dir)
print("Current Directory: ", os.getcwd())

In [ ]:
# 2) Install dependencies
# stable-baselines3는 PyPI에서 설치합니다 (이 repo에는 sb3 소스 포크가 없음).
# 로컬에서 검증된 버전 조합으로 고정합니다.
!apt-get -qq install -y libosmesa6 libgl1-mesa-glx  # osmesa(CPU) 렌더링용
!pip install -q "stable-baselines3==2.3.0" "gymnasium==0.29.1" "mujoco==3.8.0" "numpy<2" "imageio[ffmpeg]" tensorboard pygments

# Colab 기본 numpy(2.x)는 gymnasium 0.29 와 충돌하므로 numpy<2 로 낮춥니다.
# numpy 는 런타임 시작 시 이미 import돼 있어, 새 버전 적용을 위해 "최초 1회"만 자동 재시작합니다.
import numpy as _np, os as _os, sys as _sys, time as _time
if _np.__version__.startswith("2"):
    print("=" * 64)
    print("[정상] 위의 빨간 ERROR 들은 Colab 기본 패키지(jax/opencv 등) 경고라 무시해도 됩니다.")
    print("설치 완료 - numpy<2 적용을 위해 런타임을 \"1회만\" 자동 재시작합니다 (크래시 아님).")
    print("다시 연결되면 [런타임 > 모두 실행] 을 한 번 더 누르면 그대로 끝까지 진행됩니다.")
    print("=" * 64)
    _sys.stdout.flush()
    _time.sleep(1.5)            # 안내문이 화면에 보이도록 잠시 대기
    _os.kill(_os.getpid(), 9)   # 런타임 재시작

In [ ]:
# 3) 의존성 설치 '후' 환경 설정 + numpy 패치
#    (설치를 numpy import보다 먼저 했으므로 런타임 재시작이 필요 없음)
import os, sys
import yaml

sys.path.insert(0, os.path.join(repo_dir, "src"))
os.environ["MUJOCO_GL"] = "osmesa"  # Colab EGL 크래시 회피: CPU 소프트웨어 렌더링

# numpy 1.x / 2.x 호환성 패치:
# 저장된 모델이 numpy 2.x (numpy.core_ 경로) 로 직렬화된 경우를 위해
# numpy.core_ 를 numpy.core 의 alias 로 등록합니다.
import numpy, numpy.core, numpy.core.numeric, numpy.core.multiarray
import numpy.random._pickle as _np_pickle

sys.modules['numpy.core_'] = numpy.core
sys.modules['numpy.core_.numeric'] = numpy.core.numeric
sys.modules['numpy.core_.multiarray'] = numpy.core.multiarray

_orig_bg_ctor = _np_pickle.__bit_generator_ctor
def _patched_bg_ctor(bg='MT19937'):
    return bg() if isinstance(bg, type) else _orig_bg_ctor(bg)
_np_pickle.__bit_generator_ctor = _patched_bg_ctor

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import HtmlFormatter
import inspect

def _render_code(code, title="code", max_height=400, bg="transparent", indent=16):
    style_name = "native" if in_colab else "friendly"
    formatter = HtmlFormatter(style=style_name, noclasses=True, linenos="inline")
    html = highlight(code, PythonLexer(), formatter)
    css = """
    <style>
    .highlight pre { margin: 0; text-align: left; }
    </style>
    """
    return HTML(f"""
    {css}
    <details>
      <summary>{title}</summary>
      <div style="margin-top:8px; margin-left:{indent}px; max-height:{max_height}px; overflow:auto; border:1px solid #ddd; padding:10px; background:{bg};">
        {html}
      </div>
    </details>
    """)


def show_code(path, max_height=400, bg="transparent"):
    code = Path(path).read_text()
    return _render_code(code, title=str(path), max_height=max_height, bg=bg)

def show_func(obj, max_height=400, bg="transparent"):
    code = inspect.getsource(obj)
    return _render_code(code, max_height=max_height, bg=bg)

---

## 1. 설정 파일 살펴보기

학습에 사용하는 주요 설정/구현 파일을 노트북에서 바로 열어봅니다.

- **`src/params.yaml`** — PPO 하이퍼파라미터, 병렬 환경 수, 총 timestep, 평가 주기 등 학습 설정
- **`src/envs.yaml`** — 보상/패널티 가중치, 명령 속도 범위, 보행(gait) 패턴, 종료 조건
- **`src/mdp/reward.py`** — 위 가중치가 실제로 계산되는 보상 함수 구현


In [ ]:
# 설정/구현 파일 펼쳐 보기 (각 제목을 클릭하면 코드가 펼쳐집니다)
display(show_code(f"{repo_dir}/src/params.yaml"))
display(show_code(f"{repo_dir}/src/envs3.yaml"))
display(show_code(f"{repo_dir}/src/mdp/reward.py", max_height=600))

---

## 2. Go2 MuJoCo 환경 생성

`Go2MujocoEnv`(position 제어 전용, `unitree_go2/scene_position.xml`)를 생성하고
관측(observation)·행동(action) 공간을 확인합니다. `params.yaml` 도 함께 로드합니다.


In [ ]:
import numpy as np
import src.go2_mujoco_env as go2_env

# 학습 설정 로드
policy_cfg_path = f"{repo_dir}/src/params.yaml"
with open(policy_cfg_path, "r", encoding="utf-8") as f:
    policy_cfg = yaml.safe_load(f)

# 환경 생성 및 공간 확인
env = go2_env.Go2MujocoEnv(prj_path=repo_dir, render_mode=None)
obs, info = env.reset()
print("n_envs       :", policy_cfg["n_envs"])
print("batch_size   :", policy_cfg["policy"]["batch_size"])
print("Observation shape:", np.array(obs).shape)
print("Action space     :", env.action_space)
print("Observation space:", env.observation_space)
env.close()

---

## 3. Pretrained 모델 불러와 4,800 step 추가 학습

전체 학습은 Colab 에서 비용이 크므로, **env3 모델을 불러와
`reset_num_timesteps=False` 로 4,800 step 만 이어서 학습**합니다.

- `PRETRAINED_MODEL_PATH` : 불러올 모델 경로. 기본값은
  `models/pretrained_env3/best_model.zip` 입니다. 다른 모델로 바꾸려면
  이 한 줄만 수정하면 됩니다.
- `ENV_CFG_PATH` : 학습에 사용할 보상 설정. 기본값 `src/envs3.yaml`.
- `ADDITIONAL_TIMESTEPS` : 추가 학습 step (기본 4,800)
- 학습 결과는 `models/<날짜시각>-finetune10k/` 에 저장됩니다
  (`best_model.zip`, 체크포인트, `final_model.zip`).
- numpy 버전 불일치로 인한 로드 실패를 막기 위해 `PPO.load(custom_objects=...)` 를 사용합니다.


In [ ]:
import time, gc
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import (
    EvalCallback, CheckpointCallback, CallbackList,
)
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv

import src.go2_mujoco_env as go2_env
from src.utils.reward_logging_callback import RewardLoggingCallback

# ===== 설정 (필요시 수정) =====
# env3 보상으로 학습한 모델을 출발점으로 사용.
PRETRAINED_MODEL_PATH = f"{repo_dir}/models/pretrained_env3/best_model.zip"
ADDITIONAL_TIMESTEPS = 4800
N_ENVS = 4  # Colab 메모리 안전값 (12는 OOM). 더 빠르게 하려면 ADDITIONAL_TIMESTEPS를 줄이세요
SEED = policy_cfg["seed"]

assert os.path.exists(PRETRAINED_MODEL_PATH), (
    f"pretrained 모델을 찾을 수 없습니다: {PRETRAINED_MODEL_PATH}\n"
    f"PRETRAINED_MODEL_PATH 를 존재하는 .zip 경로로 수정하세요."
)

model_dir = f"{repo_dir}/models"
log_dir = f"{repo_dir}/logs"
os.makedirs(model_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

ENV_CFG_PATH = f"{repo_dir}/src/envs3.yaml"   # 이 보상 설정으로 학습
train_env_kwargs = {"prj_path": repo_dir, "cfg_path": ENV_CFG_PATH}
vec_env = make_vec_env(
    go2_env.Go2MujocoEnv, env_kwargs=train_env_kwargs,
    n_envs=N_ENVS, seed=SEED, vec_env_cls=SubprocVecEnv,
)
eval_env = make_vec_env(
    go2_env.Go2MujocoEnv, env_kwargs=train_env_kwargs,
    n_envs=1, seed=SEED + 10_000, vec_env_cls=DummyVecEnv,
)

run_name = time.strftime("%Y-%m-%d_%H-%M-%S") + "-finetune10k"
model_path = f"{model_dir}/{run_name}"
os.makedirs(model_path, exist_ok=True)
print("저장 위치:", model_path)

# numpy 버전 불일치로 space 역직렬화가 실패할 수 있어 현재 env 의 space 로 대체
_dummy = go2_env.Go2MujocoEnv(prj_path=repo_dir, cfg_path=ENV_CFG_PATH, render_mode=None)
custom_objects = {
    "observation_space": _dummy.observation_space,
    "action_space": _dummy.action_space,
    "lr_schedule": lambda _: policy_cfg["policy"]["learning_rate"],
    "clip_range": lambda _: policy_cfg["policy"]["clip_range"],
}
_dummy.close()

checkpoint_callback = CheckpointCallback(
    save_freq=max(policy_cfg["policy"]["n_steps"] * policy_cfg["log"]["interval"] // N_ENVS, 1),
    save_path=model_path, name_prefix="model",
    save_replay_buffer=False, save_vecnormalize=False,
)
eval_callback = EvalCallback(
    eval_env, best_model_save_path=model_path, log_path=log_dir,
    eval_freq=max(policy_cfg["eval_freq"] // N_ENVS, 1),
    n_eval_episodes=5, deterministic=True, render=False,
)
callbacks = CallbackList([eval_callback, checkpoint_callback, RewardLoggingCallback()])

print(f"Loading pretrained model from {PRETRAINED_MODEL_PATH}")
model = PPO.load(
    PRETRAINED_MODEL_PATH, env=vec_env, custom_objects=custom_objects,
    verbose=1, tensorboard_log=log_dir,
)
# 학습률을 새로 적용 (스케줄 재생성)
model.learning_rate = policy_cfg["policy"]["learning_rate"]
model._setup_lr_schedule()

model.learn(
    total_timesteps=ADDITIONAL_TIMESTEPS,
    reset_num_timesteps=False,   # pretrained 의 step 카운트를 이어감
    progress_bar=True,
    tb_log_name=run_name,
    callback=callbacks,
)
model.save(f"{model_path}/final_model")
print("최종 모델 저장:", f"{model_path}/final_model.zip")

vec_env.close()
eval_env.close()
del model
gc.collect()

---

## 4. 학습한 모델로 추론 → 보행 영상 생성

위에서 추가 학습해 저장한 모델을 불러와, 직진 command `[vx, vy, wz] = [0.9, 0, 0]`
으로 롤아웃하면서 프레임을 모아 mp4 로 저장합니다. (제어 50Hz, 영상 10FPS)


In [ ]:
import imageio
from tqdm.auto import tqdm
from stable_baselines3 import PPO
import src.go2_mujoco_env as go2_env

# best_model 우선, 없으면 final_model 사용
eval_model_path = f"{model_path}/best_model.zip"
if not os.path.exists(eval_model_path):
    eval_model_path = f"{model_path}/final_model.zip"
print("추론 모델:", eval_model_path)

given_command = [0.9, 0.0, 0.0]   # [vx (m/s), vy (m/s), wz (rad/s)]
WIDTH, HEIGHT = 320, 240

env = go2_env.Go2MujocoEnv(
    prj_path=repo_dir,
    cfg_path=ENV_CFG_PATH,
    given_command=given_command,
    render_mode="rgb_array",
    camera_name="tracking",
    width=WIDTH, height=HEIGHT,
)
env._reset_noise_scale = 0.05  # 초기 노이즈 축소

custom_objects = {
    "observation_space": env.observation_space,
    "action_space": env.action_space,
    "lr_schedule": lambda _: policy_cfg["policy"]["learning_rate"],
    "clip_range": lambda _: policy_cfg["policy"]["clip_range"],
}
model = PPO.load(eval_model_path, env=env, custom_objects=custom_objects, verbose=0)

# control rate = 50 Hz
max_time_step_s = policy_cfg["test"]["max_time_step_s"]
video_fps = 10
render_interval = 50 // video_fps
max_steps = int(max_time_step_s * 50)
video_path = f"{model_path}/rollout_{run_name}.mp4"
print("max time:", max_time_step_s, " max steps:", max_steps)

obs, _ = env.reset()
frames = []
ep_len, ep_reward = 0, 0.0
for step in tqdm(range(max_steps), desc="rollout", unit="step"):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    ep_len += 1
    ep_reward += reward
    if step % render_interval == 0:
        frames.append(env.render())
    if terminated or truncated:
        print(f"episode finished: ep_len={ep_len}, ep_reward={ep_reward:.3f}")
        obs, _ = env.reset()
        ep_len, ep_reward = 0, 0.0
env.close()

imageio.mimwrite(
    video_path, frames, fps=video_fps,
    codec="libx264", quality=8, pixelformat="yuv420p",
)
print("영상 저장:", video_path)

---

## 5. 영상 재생

생성된 mp4 를 노트북에서 바로 재생합니다.


In [ ]:
from IPython.display import Video, display

display(Video(video_path, embed=True, html_attributes="controls autoplay loop"))

---

## 6. TensorBoard 로 학습 로그 보기

PPO 학습 중 기록된 보상/손실 곡선을 TensorBoard 로 확인합니다.
런(run)별로 `logs/` 아래에 저장되며, 셀을 실행하면 노트북 안에 대시보드가 뜹니다.


In [ ]:
# 학습 로그 시각화 (Colab 인라인 TensorBoard)
import os
os.chdir(repo_dir)          # logs 상대경로 기준 보장
%load_ext tensorboard
%tensorboard --logdir logs